
# MiniMax H3 — 2×T4 Comfy Generator (public URL enabled)

Smoke test is real: T2V workflow + MultiStream on Comfy.
After everything is verified, this prints the public Base URL + API Key.


In [ ]:

import os
from pathlib import Path
os.environ.setdefault("HF_HUB_DISABLE_XET","1")
os.environ.setdefault("HF_HUB_DISABLE_HF_TRANSFER","1")
os.environ.setdefault("HF_HOME","/tmp/hf")
os.environ.setdefault("TORCH_HOME","/tmp/torch")
os.environ.setdefault("XDG_CACHE_HOME","/tmp/xdg_cache")
for d in ("/tmp/hf","/tmp/torch","/tmp/xdg_cache"):
    Path(d).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
try:
    from kaggle_secrets import UserSecretsClient
    tok=UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"]=tok
    print("HF_TOKEN loaded")
except Exception:
    print("HF_TOKEN from secrets not set; continuing where possible")


In [ ]:

import shutil, subprocess, psutil, torch, os, sys
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); f,t=torch.cuda.mem_get_info(i)
    print(f"GPU{i}: {p.name} {t/2**30:.1f}GB free={f/2**30:.1f}GB")
ram=psutil.virtual_memory()
print(f"RAM total={ram.total/2**30:.1f}GB avail={ram.available/2**30:.1f}GB")
print("/tmp free:", shutil.disk_usage('/tmp').free/2**30, "GiB")
print(subprocess.run(["nvidia-smi","topo","-m"],capture_output=True,text=True).stdout)


In [ ]:

import subprocess,sys,os
CTE="/tmp/ComfyUI"
if not os.path.isdir(CTE):
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",CTE],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(CTE,"requirements.txt")],check=True)
cn=os.path.join(CTE,"custom_nodes","ComfyUI-H3-MultiStream")
if not os.path.isdir(cn):
    subprocess.run(["git","clone","--depth","1","https://github.com/martonsagi/Comfy-H3-MultiStream.git",cn],check=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(cn,"requirements.txt")],check=False)
except Exception:
    pass
print("ComfyUI at",CTE)


In [ ]:

import subprocess,sys,os
from pathlib import Path
CTE="/tmp/ComfyUI"
EXPECT=[
 "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
 "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
 "vae/minimax_h3_video_vae_int8_convrot.safetensors",
 "vae/minimax_h3_audio_vae_fp32.safetensors",
 "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
]
def missing():
    out=[]
    for f in EXPECT:
        p=Path(CTE)/"models"/f
        if not p.exists() or p.stat().st_size==0: out.append(f)
    return out
miss=missing()
if miss:
    print("downloading",len(miss),"files from Comfy-Org/MiniMax-H3")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Comfy-Org/MiniMax-H3",allow_patterns=miss,local_dir=os.path.join(CTE,"models"))
bad=[]
for f in EXPECT:
    p=Path(CTE)/"models"/f
    sz=p.stat().st_size if p.exists() else 0
    print(("OK  " if sz else "MISS"),sz,f)
    if not sz: bad.append(f)
if bad: raise RuntimeError("model files missing after download: %s"%bad)
subprocess.run(["curl","-L","-sSf","-o","/tmp/h3_t2v.json","https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json"],check=True)
print("assets staged in /tmp/ComfyUI/models, workflow at /tmp/h3_t2v.json")


In [ ]:

import json
wf=json.load(open("/tmp/h3_t2v.json"))
subs=(wf.get("definitions") or {}).get("subgraphs",[])
print("workflow nodes:",len(wf.get("nodes",[])),"subgraphs:",[s.get("name") for s in subs])
print("H3 MultiStream insertion + subgraph flattening happen in the submit cell")


In [ ]:

import subprocess,sys,os,time,json
c=subprocess.Popen([sys.executable,"/tmp/ComfyUI/main.py","--listen","127.0.0.1","--port","8188","--preview-method","latent2rgb"],
                   cwd="/tmp/ComfyUI", stdout=open("/tmp/comfy.log","w"), stderr=subprocess.STDOUT)
print("Comfy pid",c.pid)
ready=False
for i in range(72):
    try:
        import urllib.request as ur
        ur.urlopen("http://127.0.0.1:8188/history", timeout=3)
        ready=True; break
    except Exception: time.sleep(5)
if not ready:
    raise RuntimeError("ComfyUI did not listen in time")
print("ComfyUI local ready")


In [ ]:

import json,time,urllib.request,urllib.error,os
BASE="http://127.0.0.1:8188"
wf=json.load(open("/tmp/h3_t2v.json"))
info=json.loads(urllib.request.urlopen(BASE+"/object_info").read())
MISSING=object()

def schema(ct):
    if ct not in info:
        raise RuntimeError("node type not registered: "+ct)
    s=info[ct]["input"]
    return set(s.get("required",{}))|set(s.get("optional",{}))

subs={s["id"]:s for s in ((wf.get("definitions") or {}).get("subgraphs") or [])}
main_links={l[0]:l for l in (wf.get("links") or [])}
instance_out={}
prompt={}
_next=[1000]
def alloc():
    _next[0]+=1
    return str(_next[0])

def main_value(link_id):
    l=main_links[link_id]
    src,slot=l[1],l[2]
    if str(src) in instance_out:
        return instance_out[str(src)](slot)
    return [str(src),slot]

def expand_instance(node):
    sub=subs[node["type"]]
    ilinks={l["id"]:l for l in (sub.get("links") or [])}
    inst_inputs={i["name"]:i for i in (node.get("inputs") or [])}
    wnamed=node.get("widgets_values_named") or {}

    def sub_input(slot):
        name=sub["inputs"][slot]["name"]
        ii=inst_inputs.get(name)
        if ii is not None and ii.get("link") is not None:
            return main_value(ii["link"])
        if name in wnamed:
            return wnamed[name]
        return MISSING

    idmap={}
    for n in sub["nodes"]:
        schema(n["type"])
        idmap[str(n["id"])]=alloc()

    def out_value(slot):
        out=(sub.get("outputs") or [])[slot]
        lk=ilinks[out["linkIds"][0]]
        return [idmap[str(lk["origin_id"])],lk["origin_slot"]]

    for n in sub["nodes"]:
        valid=schema(n["type"])
        linked=set(); inputs={}
        for inp in (n.get("inputs") or []):
            nm=inp["name"]
            if inp.get("link") is None: continue   # links always included: autogrow slots
            linked.add(nm)                          # (values.a) only exist after finalization
            lk=ilinks.get(inp["link"])
            if lk is None: raise RuntimeError("inner link missing: %s"%inp["link"])
            if lk["origin_id"]==-10:
                v=sub_input(lk["origin_slot"])
            else:
                oid=idmap.get(str(lk["origin_id"]))
                if oid is None: raise RuntimeError("inner origin missing: %s"%lk["origin_id"])
                v=[oid,lk["origin_slot"]]
            if v is MISSING: continue
            inputs[nm]=v
        named=n.get("widgets_values_named") or {}
        for nm in valid:
            if nm in inputs or nm in linked: continue
            if nm in named: inputs[nm]=named[nm]
        prompt[idmap[str(n["id"])]]={"class_type":n["type"],"inputs":inputs}
    return out_value

# pass 1: flatten every subgraph instance
for n in wf.get("nodes",[]):
    if n["type"] in subs:
        instance_out[str(n["id"])]=expand_instance(n)
        print("expanded subgraph",n["type"],"-> nodes:",len(prompt))

# pass 2: regular (UI graph) nodes
for n in wf.get("nodes",[]):
    t=n["type"]
    if t in subs: continue
    if t not in info:
        print("skipping UI-only node:",t,"#%s"%n["id"]); continue
    valid=schema(t)
    linked=set(); inputs={}
    for inp in (n.get("inputs") or []):
        nm=inp["name"]
        if inp.get("link") is None: continue
        linked.add(nm)
        v=main_value(inp["link"])
        if v is MISSING: continue
        inputs[nm]=v
    named=n.get("widgets_values_named") or {}
    for nm in valid:
        if nm in inputs or nm in linked: continue
        if nm in named: inputs[nm]=named[nm]
    prompt[str(n["id"])]={"class_type":t,"inputs":inputs}

print("flattened prompt nodes:",len(prompt))

# ~1K quality target: official 768p table row (1344x768)
for nid,node in prompt.items():
    if node["class_type"]=="ResolutionSelector" and "megapixels" in node["inputs"]:
        node["inputs"]["megapixels"]=0.98
        print("ResolutionSelector megapixels -> 0.98 (1344x768)")

# insert H3MultiStream after the model switch, feeding BasicGuider + BasicScheduler
schema("H3MultiStream")
targets=[]
for nid,node in prompt.items():
    if node["class_type"] in ("BasicGuider","BasicScheduler") and isinstance(node["inputs"].get("model"),list):
        targets.append((nid,node["inputs"]["model"]))
if not targets:
    raise RuntimeError("no BasicGuider/BasicScheduler with a linked model input")
src=targets[0][1]
h3_in={"model":src,"enabled":True,"second_gpu":-1,"exchange":"host","exchange_chunks":8,
       "sparse_attention":False,"dynamic_vram":"keep","vram_block_cache":False,
       "vram_reserve_gb":2.0,"sparse_vsa":False,"weight_cache":False,"cache_ram_reserve_gb":0.0}
valid=schema("H3MultiStream")
missing=[k for k in valid if k not in h3_in and k!="gpus" and not isinstance(h3_in.get(k),list)]
h3_in={k:v for k,v in h3_in.items() if k in valid}
prompt["9999"]={"class_type":"H3MultiStream","inputs":h3_in}
n_p=0
for nid,v in targets:
    if v==src:
        prompt[nid]["inputs"]["model"]=["9999",0]; n_p+=1
print("H3MultiStream inserted as node 9999, feeding",n_p,"consumers; model source",src)
if missing: print("WARNING: H3 inputs not supplied:",missing)

body={"prompt":prompt,"extra_data":{"preview_method":"latent2rgb"}}
try:
    r=urllib.request.Request(BASE+"/prompt",data=json.dumps(body).encode(),headers={"Content-Type":"application/json"})
    resp=urllib.request.urlopen(r,timeout=60).read().decode()
    print("submitted:",resp)
except urllib.error.HTTPError as ee:
    print("prompt failed",ee,"body:",ee.read().decode()[:6000])
    raise


In [ ]:

import json,time,urllib.request,os
pid_j=json.loads(resp)
pid=pid_j.get("prompt_id") or pid_j.get("id") or pid_j.get("prompt")
print("prompt_id",pid)
got=None
for i in range(360):
    try:
        h=json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}",timeout=5).read())
        if h and h.get(pid):
            got=h.get(pid); break
    except Exception: pass
    if i%12==0:
        try:
            last=[l for l in open("/tmp/comfy.log",errors="ignore").read().splitlines() if l.strip()][-1:]
            print(f"[{i*10}s]", last[0][:200] if last else "(no log yet)")
        except Exception: pass
    time.sleep(10)
if not got: raise RuntimeError("smoke did not finish within 60 min")
st=got.get("status",{})
print("smoke status:",st.get("status_str"),"completed:",st.get("completed"),"/",st.get("remaining"))
if st.get("status_str")=="error": raise RuntimeError("smoke run had execution errors: "+json.dumps(st)[:2000])
print("smoke finished; outputs:")
for root,_,files in os.walk("/tmp/ComfyUI/output"):
    for f in files:
        if f.lower().endswith((".mp4",".webm",".mov",".webp",".gif",".png")):
            print("found:",os.path.join(root,f))


In [ ]:

import subprocess,time,re,urllib.request,json,os
from pathlib import Path
Path("/tmp/h3_api").mkdir(parents=True,exist_ok=True)
ok=False
for u in ["https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
          "https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64"]:
    try:
        data=urllib.request.urlopen(u,timeout=120).read()
        if len(data)>1000000:
            open("/tmp/h3_api/cloudflared","wb").write(data); ok=True; break
        print("cf dl too small from",u,len(data))
    except Exception as e:
        print("cf dl failed:",u,e)
if not ok: raise RuntimeError("cloudflared download failed")
os.chmod("/tmp/h3_api/cloudflared",0o755)
log=open("/tmp/h3_api/cf.log","w")
proc=subprocess.Popen(["/tmp/h3_api/cloudflared","tunnel","--url","http://127.0.0.1:8188"],stdout=log,stderr=subprocess.STDOUT)
url=None; pat=re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for i in range(90):
    t=open("/tmp/h3_api/cf.log","r",encoding="utf-8",errors="ignore").read()
    m=pat.search(t)
    if m:
        try:
            r=urllib.request.urlopen(m.group(0)+"/history",timeout=5)
            if r.status==200: url=m.group(0); break
        except Exception: pass
    time.sleep(2)
if not url: raise RuntimeError("cloudflare tunnel not healthy")
print("Verified public endpoint:",url)
print("\n================ MINIMAX H3 — GENERATION READY ================")
print("Base URL:"); print("  "+url)
print("API prompt endpoint:"); print("  "+url+"/prompt")
print("Example:")
print("===============================================================")
